# Propionic Acid

In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
propionic,74.0785,3.01515,3.205896845,200.6790442,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
propionic,H,propionic,e,2000.0,0.03
"""

model = PCSAFT(["propionic"], userlocations = [like_parameter, assoc_parameter])

PCSAFT{BasicIdeal, Float64} with 1 component:
 "propionic"
Contains parameters: Mw, segment, sigma, epsilon, epsilon_assoc, bondvol

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_propionic.csv")
fix_line_endings("rhol_propionic.csv")

Fixed: satp_propionic.csv
Fixed: rhol_propionic.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end 

my_liquid_density (generic function with 1 method)

In [5]:
toestimate = [
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 500.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.001,
        :upper   => 0.50,
        :guess   => 0.4
    ),
]

2-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 500.0)
 Dict(:upper => 0.5, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.001)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_propionic.csv",
        "satp_propionic.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 0.03901567490766578


In [7]:
method = ECA(; options = Options(iterations = 10000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([2743.4247225017098, 0.12545098937464], PCSAFT{BasicIdeal, Float64}("propionic"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_propionic.csv",   my_saturation_p)


=== AAD: satp_propionic.csv ===


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


Clapeyron Estimator  exp           calc          ARD%    
352.5500    9919.008000   9987.008025   0.6856  
353.1500    10198.980000  10263.564269  0.6332  
353.4500    10358.964000  10404.289454  0.4375  
356.6500    12052.128000  12011.392068  0.3380  
358.9500    13265.340000  13293.766422  0.2143  
359.1500    13398.660000  13410.609494  0.0892  
364.6500    16838.316000  16986.415107  0.8795  
368.3500    19744.692000  19824.163949  0.4025  
368.7500    20131.320000  20153.764333  0.1115  
372.0500    23144.352000  23055.063208  0.3858  
372.6500    23557.644000  23618.886534  0.2600  
373.9500    26237.376000  24880.694112  5.1708  
375.4500    26717.328000  26407.099856  1.1611  
378.5500    30210.312000  29813.914409  1.3121  
381.5500    33276.672000  33456.877438  0.5415  
383.9500    36463.020000  36634.830057  0.4712  
386.2500    39716.028000  39914.892054  0.5007  
388.7500    43488.984000  43756.422361  0.6150  
390.7500    46688.664000  47048.535770  0.7708  
392.5500   

0.7796625617116268

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_propionic.csv",   my_liquid_density)


=== AAD: rhol_propionic.csv ===
Clapeyron Estimator  exp           calc          ARD%    
273.1500    1015.030000   1002.632645   1.2214  
288.1500    998.740000    988.256852    1.0496  
303.1500    982.600000    973.929412    0.8824  
288.0000    998.800000    988.400179    1.0412  
288.3000    998.700000    988.113531    1.0600  
291.8500    994.830000    984.722686    1.0160  
293.1500    993.000000    983.481321    0.9586  
AARD = 1.0327%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


1.0327495377960336